In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1. ウィンドウ関数（分析関数）

ウィンドウ関数（Window Function）は、`FROM句`の結果セットを維持したまま、グループごとの集計や順位を計算し、その結果を新しいカラムとして追加する関数である。

集計関数との大きな違いは、次のとおりである。

- 集計関数（`GROUP BY`）：複数の行を1つの結果にまとめる。
- ウィンドウ関数：複数の行を参照して計算するが、元の行はそのまま維持する。<br></br>
- 集計関数は、`GROUP BY`でグループ化した集計結果を取得する。
- ウィンドウ関数は、`PARTITION BY`で分けたグループの計算結果を、元の各行に追加して取得する。

#### ※ ウィンドウ関数が登場する前は、以下の2つの結果を同時に取得するために、次のようにJOINを使用していた。

In [2]:
%%sql

SELECT rating,
    AVG(length) AS avg_length
FROM film
GROUP BY rating;

rating,avg_length
PG,112.0052
G,111.0506
NC-17,113.2286
PG-13,120.4439
R,118.6615


In [18]:
%%sql

SELECT title, length, rating
FROM film

LIMIT 7;

title,length,rating
ACADEMY DINOSAUR,86,PG
ACE GOLDFINGER,48,G
ADAPTATION HOLES,50,NC-17
AFFAIR PREJUDICE,117,G
AFRICAN EGG,130,G
AGENT TRUMAN,169,PG
AIRPLANE SIERRA,62,PG-13


In [17]:
%%sql

SELECT f.title, f.rating, f.length, t.avg_length
FROM film f INNER JOIN (SELECT rating,
					AVG(length) AS avg_length
					FROM film
					GROUP BY rating) t
ON f.rating = t.rating
ORDER BY f.rating

LIMIT 7;

title,rating,length,avg_length
ACE GOLDFINGER,G,48,111.0506
AFFAIR PREJUDICE,G,117,111.0506
AFRICAN EGG,G,130,111.0506
ALAMO VIDEOTAPE,G,126,111.0506
AMISTAD MIDSUMMER,G,85,111.0506
ANGELS LIFE,G,74,111.0506
ANNIE IDENTITY,G,86,111.0506


#### ※ 
MySQL 8.0以降で追加されたウィンドウ関数を使用すると、`JOIN`を使わずに取得できる。

In [16]:
%%sql

SELECT title,
       rating,
       length,
       AVG(length) OVER(PARTITION BY rating) AS avg_length
FROM film

LIMIT 7;

title,rating,length,avg_length
ACE GOLDFINGER,G,48,111.0506
AFFAIR PREJUDICE,G,117,111.0506
AFRICAN EGG,G,130,111.0506
ALAMO VIDEOTAPE,G,126,111.0506
AMISTAD MIDSUMMER,G,85,111.0506
ANGELS LIFE,G,74,111.0506
ANNIE IDENTITY,G,86,111.0506


`GROUP BY`はグループごとの集計結果のみを表示するが、`ウィンドウ関数`は元の行と集計結果を一緒に表示できる。

従来のSQLでは、次のような処理を行うために複雑なクエリが必要だった。

- 順位を求める
- 前の行や次の行と比較する
- 累積合計を計算する
- 移動平均を計算する
- グループごとの順位を求める

ウィンドウ関数を使用すると、これらの処理を元の行を維持したまま簡単に実行できる。

#### ※ `SELECT句`にサブクエリがある場合、サブクエリの結果とメインクエリの結果を結合して取得する。

In [11]:
%%sql

SELECT name, (SELECT 10 AS val)
FROM category;

name,(SELECT 10 AS val)
Action,10
Animation,10
Children,10
Classics,10
Comedy,10
Documentary,10
Drama,10
Family,10
Foreign,10
Games,10


このクエリは、内部的には次のように実行される

In [10]:
%%sql

SELECT name, t.val
FROM category
CROSS JOIN (SELECT 10 AS val) t;

name,val
Action,10
Animation,10
Children,10
Classics,10
Comedy,10
Documentary,10
Drama,10
Family,10
Foreign,10
Games,10


→ ウィンドウ関数も同様である。ウィンドウ関数の結果セットとメインクエリの間で結合が行われる。

In [15]:
%%sql

SELECT
    title,
    rating,
    length,  -- メインクエリの結果セット
    AVG(length) OVER (
        PARTITION BY rating
    ) AS avg_length  -- ウィンドウ関数の計算結果
FROM film

LIMIT 7;

title,rating,length,avg_length
ACE GOLDFINGER,G,48,111.0506
AFFAIR PREJUDICE,G,117,111.0506
AFRICAN EGG,G,130,111.0506
ALAMO VIDEOTAPE,G,126,111.0506
AMISTAD MIDSUMMER,G,85,111.0506
ANGELS LIFE,G,74,111.0506
ANNIE IDENTITY,G,86,111.0506


## 2. ウィンドウ関数の構文

```sql
関数() OVER (
    PARTITION BY オプション
    ORDER BY オプション
    ROWS / RANGE オプション
)
```

#### 1) OVER()

すべてのウィンドウ関数には`OVER()`が必要である。

#### 2) PARTITION BY

`GROUP BY`と最も似た役割を持つが、元の結果セットはそのまま維持し、集計のための別の計算グループを作成する。

#### 3) ORDER BY

集計関数を実行する前に並べ替えを行う。

例：`id`順に並べ替えた後に累積合計を求める、成績順に並べ替えた後に順位を求める、など

#### 4) ROWS / RANGE

`ORDER BY`オプションがある場合、パーティションをさらに分割してウィンドウを指定できる。

## 3. ウィンドウ関数の種類

| 分類 | 関数 |
|---|---|
| 順位関数 | `ROW_NUMBER()`、`RANK()`、`DENSE_RANK()`、`NTILE()` |
| 行参照関数 | `LAG()`、`LEAD()`、`FIRST_VALUE()`、`LAST_VALUE()`、`NTH_VALUE()` |
| 集計関数 | `SUM()`、`AVG()`、`COUNT()`、`MAX()`、`MIN()` |
| 分布関数 | `CUME_DIST()`、`PERCENT_RANK()` |

#### 1) ROW_NUMBER()

- 各行に`1、2、3、...`の連番を付ける。
- 同じ値があっても、番号は重複しない。

In [30]:
%%sql

SELECT
    title,
    ROW_NUMBER() OVER(ORDER BY length DESC) AS `row_number`
FROM film

LIMIT 5;

title,row_number
CHICAGO NORTH,1
CONTROL ANTHEM,2
DARN FORRESTER,3
GANGS PRIDE,4
HOME PITY,5


#### ※ `PARTITION BY`を省略すると、全体が1つのパーティションになる。???

In [28]:
%%sql

SELECT
    title, rating,
    ROW_NUMBER() OVER(PARTITION BY rating ORDER BY length DESC) r
FROM film
ORDER BY r

LIMIT 10;

title,rating,r
CONTROL ANTHEM,G,1
WORST BANGER,PG,1
CHICAGO NORTH,PG-13,1
HOME PITY,R,1
CRYSTAL BREAKING,NC-17,1
DARN FORRESTER,G,2
MONSOON CAUSE,PG,2
GANGS PRIDE,PG-13,2
SOLDIERS EVOLUTION,R,2
KING EVOLUTION,NC-17,2


#### 2) RANK()

- 順位を計算する。
- 同じ値には同じ順位を付ける。
- 次の順位は飛ばされる。

例：`1、2、2、4`

In [36]:
%%sql

SELECT
    title,
    length,
    RANK() OVER(ORDER BY length DESC)
FROM film

LIMIT 7;

title,length,RANK() OVER(ORDER BY length DESC)
CHICAGO NORTH,185,1
CONTROL ANTHEM,185,1
DARN FORRESTER,185,1
GANGS PRIDE,185,1
HOME PITY,185,1
MUSCLE BRIGHT,185,1
POND SEATTLE,185,1


#### 3) DENSE_RANK()

- 順位を計算する。
- 同じ値には同じ順位を付ける。
- 次の順位を飛ばさない。

例：`1、2、2、3`

In [37]:
%%sql

SELECT
    title,
    length,
    DENSE_RANK() OVER (
        ORDER BY length DESC
    )
FROM film

LIMIT 7;

title,length,DENSE_RANK() OVER ( ORDER BY length DESC )
CHICAGO NORTH,185,1
CONTROL ANTHEM,185,1
DARN FORRESTER,185,1
GANGS PRIDE,185,1
HOME PITY,185,1
MUSCLE BRIGHT,185,1
POND SEATTLE,185,1


#### 4) NTILE() グループ分け

- データを`n`個のグループに分ける。
- 上位25%：`NTILE(4)`
- 上位10%：`NTILE(10)`
- 四分位数：`NTILE(4)`
- 分位分析などに使用される。

In [39]:
%%sql

SELECT
    title,
    length,
    NTILE(4) OVER(ORDER BY length)
FROM film

LIMIT 7;

title,length,NTILE(4) OVER(ORDER BY length)
ALIEN CENTER,46,1
IRON MOON,46,1
KWAI HOMEWARD,46,1
LABYRINTH LEAGUE,46,1
RIDGEMONT SUBMARINE,46,1
DIVORCE SHINING,47,1
DOWNHILL ENOUGH,47,1


#### 5) LAG() 前の行

- 現在の行を基準に、前の行の値を参照する。

- 現在の行より1つ前の行の値を取得する。

- 主に前日のデータ比較、前回の売上との比較、増減計算などに使用される。

In [41]:
%%sql

SELECT
    title,
    length,
    LAG(title) OVER(ORDER BY length)
FROM film

LIMIT 7;

title,length,LAG(title) OVER(ORDER BY length)
ALIEN CENTER,46,None
IRON MOON,46,ALIEN CENTER
KWAI HOMEWARD,46,IRON MOON
LABYRINTH LEAGUE,46,KWAI HOMEWARD
RIDGEMONT SUBMARINE,46,LABYRINTH LEAGUE
DIVORCE SHINING,47,RIDGEMONT SUBMARINE
DOWNHILL ENOUGH,47,DIVORCE SHINING


#### # `LAG()`の計算後に特定の行を絞り込む

In [ ]:
%%sql

select *
from (SELECT
	film_id,
	title,
	length,
	LAG(title) OVER(ORDER BY length)
	FROM film) t
where t.film_id = 469;

-- 先にサブクエリで全映画を対象に`LAG()`を計算し、その後、外側のクエリで`film_id`が`469`の行だけを取得する。

film_id,title,length,LAG(title) OVER(ORDER BY length)
469,IRON MOON,46,ALIEN CENTER


#### 6) LEAD() 次の行

- 現在の行を基準に、次の行の値を参照する。

- 現在の行より次の行の値を取得する。

- 主に次回の日程、次の取引日、次の値との比較などに使用される。

In [44]:
%%sql

SELECT
    title,
    length,
    LEAD(length) OVER(ORDER BY film_id)
FROM film

LIMIT 5;

title,length,LEAD(length) OVER(ORDER BY film_id)
ACADEMY DINOSAUR,86,48
ACE GOLDFINGER,48,50
ADAPTATION HOLES,50,117
AFFAIR PREJUDICE,117,130
AFRICAN EGG,130,169


#### 7) CUME_DIST() 累積分布

- データが全体の中でどの程度の位置にあるかを計算する。
- 現在の行以下のデータが全体に占める累積割合を返す。
- 結果は`0～1`の間の値になる。

In [ ]:
%%sql

SELECT
    title,
    length,
    CUME_DIST() OVER(ORDER BY length)
FROM film

LIMIT 7;

-- 上映時間が短い映画から並べ替えた後、現在の映画の上映時間以下の映画が、全映画の何％を占めるかを計算する。

title,length,CUME_DIST() OVER(ORDER BY length)
ALIEN CENTER,46,0.005
IRON MOON,46,0.005
KWAI HOMEWARD,46,0.005
LABYRINTH LEAGUE,46,0.005
RIDGEMONT SUBMARINE,46,0.005
DIVORCE SHINING,47,0.012
DOWNHILL ENOUGH,47,0.012


#### 8) PERCENT_RANK() パーセント順位

- 現在の行のパーセント順位を返す。
- 最小値は`0`で、最大値は`1`に近い値になる。

In [48]:
%%sql

SELECT
    title,
    length,
    PERCENT_RANK() OVER(ORDER BY length)
FROM film

LIMIT 7;

title,length,PERCENT_RANK() OVER(ORDER BY length)
ALIEN CENTER,46,0.0
IRON MOON,46,0.0
KWAI HOMEWARD,46,0.0
LABYRINTH LEAGUE,46,0.0
RIDGEMONT SUBMARINE,46,0.0
DIVORCE SHINING,47,0.005005005005005005
DOWNHILL ENOUGH,47,0.005005005005005005


#### 9) 集計ウィンドウ関数

各行をそのまま維持しながら、集計結果を一緒に表示する。

`AVG() OVER()`

`SUM() OVER()`

`COUNT() OVER()`

`MIN() OVER()`

`MAX() OVER()`

In [50]:
%%sql

SELECT
	title, 
    rating,
	COUNT(length) OVER(PARTITION BY rating), 
	SUM(length) OVER(PARTITION BY rating), 
	AVG(length) OVER(PARTITION BY rating), 
	MIN(length) OVER(PARTITION BY rating), 
	MAX(length) OVER(PARTITION BY rating)
FROM film

LIMIT 7;

title,rating,COUNT(length) OVER(PARTITION BY rating),SUM(length) OVER(PARTITION BY rating),AVG(length) OVER(PARTITION BY rating),MIN(length) OVER(PARTITION BY rating),MAX(length) OVER(PARTITION BY rating)
ACE GOLDFINGER,G,178,19767,111.0506,47,185
AFFAIR PREJUDICE,G,178,19767,111.0506,47,185
AFRICAN EGG,G,178,19767,111.0506,47,185
ALAMO VIDEOTAPE,G,178,19767,111.0506,47,185
AMISTAD MIDSUMMER,G,178,19767,111.0506,47,185
ANGELS LIFE,G,178,19767,111.0506,47,185
ANNIE IDENTITY,G,178,19767,111.0506,47,185
